# Think-LIBERO + caution inject

One checkpoint: `allenai/MolmoAct2-Think-LIBERO`. Kernel **molmoact2**. Not Molmo2-ER.

The extra sentence is spliced into the official `The task is to {task}` slot. `normalize_language=False` so punctuation survives.

Depth/action **payload** tokens (`<depth_12>`, `<action_3>`, …) render as `█`. Brackets (`<depth_start>`, `<depth_end>`, `<action_output>`, …) stay. Continuous actions are vectors, not tokens — they only appear in the last pane.

If `screenshot/` is empty: `source .venv/bin/activate && python download_frames.py`

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Markdown
from PIL import Image
from think_act import (
    INJECT,
    OFFICIAL_TASK,
    ThinkLibero,
    composed_task,
)

SHOT = ROOT / "screenshot"
third_p = SHOT / "sample_agentview_rgb.png"
wrist_p = SHOT / "sample_wrist_rgb.png"
if not third_p.is_file() or not wrist_p.is_file():
    raise FileNotFoundError(
        "screenshot/ missing official frames. In the molmoact2 venv: python download_frames.py"
    )
third = Image.open(third_p).convert("RGB")
wrist = Image.open(wrist_p).convert("RGB")
task = composed_task(OFFICIAL_TASK, INJECT)
print("task (injected):")
print(task)
display(Markdown("**What it sees** (Think-LIBERO card, libero_10 / ep0 / t0)"))
display(Markdown(f"agentview `{third_p.name}` {third.size}"))
display(third)
display(Markdown(f"wrist `{wrist_p.name}` {wrist.size}"))
display(wrist)

## Load + `predict_action`

Restart the kernel if you already have Think or Molmo2-ER in VRAM.

In [ ]:
thinker = ThinkLibero()
out = thinker.act(third, wrist, inject=INJECT, normalize_language=False)
print("normalize_language", out["normalize_language"])
print("n tokens", len(out["token_ids"]))

## Full output text

`generated_token_ids` decoded with specials kept. Empty here means the card did not return ids.

In [ ]:
print(out["full_text"] or "(no generated_token_ids)")

## Tokenized → English

`█` = a depth/action/state **value** token. Brackets stay as names.

In [ ]:
print(out["boxed"] or "(no generated_token_ids)")
print("--- token names ---")
for i, (tid, name) in enumerate(zip(out["token_ids"], out["token_names"])):
    print(f"{i:4d}  {tid:6d}  {name}")

## Depth bins + actions

These are the numeric outputs (10×10 depth codes, then the 10-step EEF chunk). Not spliced into the sentence.

In [ ]:
print("depth_bins")
print(out["depth_bins"])
print("updated_cells")
print(out.get("updated_cells"))
print("actions")
print(out["actions"])